In [1]:
import random
import string

In [2]:
class Addressable:
    def __init__(self):
        self.address = '0x' + ''.join(random.choices('0123456789abcdef', k=40))

In [3]:
class User(Addressable): pass

In [9]:
vitalik = User()
satoshi = User()
merkle  = User()
shannon = User()
shamir  = User()

In [34]:
class Token(Addressable):
    def __init__(self, name, symbol): # no need for decimals
        super().__init__()
        self.name   = name
        self.symbol = symbol
        
        self.balances    = {}
        self.totalSupply = 0

    def transferFrom(self, fr, to, amount):
        assert self.balances[fr.address] >= amount
        self.balances[fr.address] -= amount
        self.balances[to.address]  = 0
        self.balances[to.address] += amount

    def _mint(self, to, amount):
        self.balances[to.address]  = 0
        self.balances[to.address] += amount
        self.totalSupply          += amount

    def _burn(self, fr, amount):
        self.balances[fr.address]  = 0
        self.balances[fr.address] -= amount
        self.totalSupply          -= amount

    def balanceOf(self, user): return self.balances[user.address]

In [73]:
DYAD = Token("DYAD Stable",   "DYAD")
USDC = Token("USD Coin",      "USDC")
WETH = Token("Wrapped Ether", "WETH")
WBTC = Token("Wrapped BTC",   "WBTC")
USDT = Token("Tether USD",    "USDT")

In [74]:
DYAD._mint(vitalik, 100)
DYAD.transferFrom(vitalik, satoshi, 20)

DYAD.balanceOf(vitalik)

80

In [102]:
class Oracle(Addressable):
    def __init__(self, asset0, asset1, price): 
        self.asset0, self.asset1 = asset0, asset1
        self.price = price
    def set_price(self, price): self.price = price
    def __str__(self): 
        return f"{self.asset0.symbol}/{self.asset1.symbol} {self.price}"

In [103]:
ETH2USD = Oracle(WETH, USDC, 2615.93)
ETH2BTC = Oracle(WETH, WBTC, 0.039)

print(str(ETH2USD))
print(str(ETH2BTC))

WETH/USDC 2615.93
WETH/WBTC 0.039


In [165]:
class MicroLend(Addressable):
    def __init__(
        self,
        lend_asset,
        collateral_assets,
        borrow_asset,
        min_collat_ratio
    ):
        self.lend_asset        = lend_asset
        self.collateral_assets = collateral_assets
        self.borrow_asset      = borrow_asset
        self.min_collat_ratio  = min_collat_ratio
        self.lend_balance   = {}
        self.collat_balance = {}
        self.borrow_balance = {}

    def lend(self, user, amount): 
        self.lend_balance[user.address]  = 0
        self.lend_balance[user.address] += amount
    
    def deposit(self, user, amount):
        self.collat_balance[user.address]  = 0
        self.collat_balance[user.address] += amount

    def withdraw(self, user, amount):
        self.collat_balance[user.address] -= amount

    def borrow(self, user, amount):
        self.borrow_balance[user.address]  = 0
        self.borrow_balance[user.address] += amount

    def repay(self, user, amount): pass

    def collat_ratio(self, user): 
        borrowed = self.borrow_balance[user.address]
        if borrowed == 0: (1<<256) - 1 # a very large uint 
        return self.collat_balance[user.address] / borrowed

    def stats(self, user):
        lend   = self.lend_balance[user.address]
        collat = self.collat_balance[user.address]
        borrow = self.borrow_balance[user.address]
        cr     = round(self.collat_ratio(user)*100, 2)
        print(
            f"{'Lend:':<14} {lend} {self.lend_asset.symbol}\n"
            f"{'Collateral:':<14} {collat}\n"
            f"{'Borrowed:':<14} {borrow} {self.borrow_asset.symbol}\n"
            f"{'Collat Ratio:':<14} {cr}%\n"
        )

In [166]:
mL = MicroLend(
    WETH, 
    [USDC, WBTC, USDT],
    DYAD,
    1.5                       # 150%
)

In [169]:
mL.lend(vitalik, 1_000)

mL.deposit(vitalik, 200)
mL.borrow (vitalik, 130)

In [170]:
mL.stats(vitalik)

Lend:          1000 WETH
Collateral:    200
Borrowed:      130 DYAD
Collat Ratio:  153.85%



# Interest Rate Equations

In [226]:
def utilization(lend, borrow): return borrow/lend

In [227]:
def borrow_apr(lend, borrow): return utilization(lend, borrow) * 0.056

In [228]:
def lend_apr(lend, borrow):
    return utilization(lend, borrow) * borrow_apr(lend, borrow) * 0.9

In [229]:
borrow_apr(100, 90)

0.0504

In [230]:
lend_apr(100, 90)

0.040824000000000006